# 🎬 Bollywood Movie Recommendation System – Data Science Pipeline (Jupyter Notebook)

This notebook implements the **Bollywood Recommender** project and clearly demonstrates
key **Data Science techniques**:

- Data loading & inspection  
- Data cleaning & preprocessing  
- Exploratory Data Analysis (EDA) with graphs  
- Text feature engineering using **TF-IDF**  
- Similarity-based recommendation using **Cosine Similarity**  

> 💡 Make sure you have the dataset file: `IMDB-Movie-Dataset(2023-1951).csv`  
> and place it inside a folder named `data/` in the same directory as this notebook.


## 1. Import Required Libraries

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

## 2. Load the Bollywood Movies Dataset

**Data Science step:** *Data Collection & Loading*

We load the CSV file containing Bollywood movie metadata.  
Update the `csv_path` below if your dataset is in a different location.


In [ ]:
# Path to the dataset (relative to this notebook)
csv_path = os.path.join("data", "IMDB-Movie-Dataset(2023-1951).csv")

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Dataset not found at: {csv_path}. Please check the path.")

movies = pd.read_csv(csv_path)
print("Dataset loaded! Shape:", movies.shape)
movies.head()

## 3. Exploratory Data Analysis (EDA) – Part 1: Structure & Quality

**Data Science techniques used here:**

- Understanding data dimensions (`shape`)  
- Inspecting sample rows (`head`)  
- Checking data types and null values (`info`, `isnull`)  
- Basic descriptive statistics (`describe`)  


In [ ]:
print("Dataset Shape:", movies.shape)
movies.head()

In [ ]:
movies.info()

In [ ]:
print("Missing values per column:")
movies.isnull().sum()

In [ ]:
# Descriptive statistics (including object columns)
movies.describe(include="all").transpose()

## 4. Data Cleaning & Preprocessing

**Data Science techniques demonstrated:**

1. **Renaming & normalizing column names**  
2. **Selecting useful features**  
3. **Handling missing values** (imputation with empty strings for text)  
4. **Feature engineering** – creating a combined text **soup** that merges genre, overview, director, and cast.


In [ ]:
# 4.1 Normalize column names (lowercase, strip spaces)
movies.columns = [c.strip().lower() for c in movies.columns]
print("Columns after normalization:")
print(movies.columns.tolist())

# 4.2 Rename columns to standard names
rename_map = {
    "movie_name": "title",
    "name": "title",
    "title": "title",
    "movie id": "movie_id",
    "movie_id": "movie_id",
    "year": "year",
    "genre": "genre",
    "genres": "genre",
    "overview": "overview",
    "synopsis": "overview",
    "description": "overview",
    "director": "director",
    "cast": "cast",
    "actors": "cast",
}

movies = movies.rename(columns=rename_map)

# 4.3 Check required columns
required = ["title", "genre", "overview"]
for col in required:
    if col not in movies.columns:
        raise ValueError(
            f"Required column '{col}' missing from CSV.\nFound columns: {movies.columns.tolist()}"
        )

# 4.4 Keep only useful columns
keep = [c for c in ["movie_id", "title", "year", "genre", "overview", "director", "cast"] if c in movies.columns]
movies = movies[keep]

print("Columns kept for model:", movies.columns.tolist())
movies.head()

In [ ]:
# 4.5 Handle missing values for text fields
for col in ["genre", "overview", "director", "cast"]:
    if col in movies.columns:
        missing_before = movies[col].isnull().sum()
        movies[col] = movies[col].fillna("")
        missing_after = movies[col].isnull().sum()
        print(f"Column: {col} | Missing before: {missing_before} | Missing after: {missing_after}")

In [ ]:
# 4.6 Feature engineering: create text 'soup' used for recommendations
def col_text(col):
    return movies[col].astype(str) if col in movies.columns else ""

movies["soup"] = (
    col_text("genre") + " " +
    col_text("overview") + " " +
    col_text("director") + " " +
    col_text("cast")
)

movies[["title", "genre", "director", "cast", "soup"]].head()

## 5. Exploratory Data Analysis (EDA) – Part 2: Visual Insights

**Data Science techniques demonstrated:**

- Univariate analysis (single variable distributions)  
- Simple visualizations using **Matplotlib**  
- Understanding patterns like most common genres & directors  


### 5.1 Number of Movies Released Per Year

In [ ]:
if "year" in movies.columns:
    movies["year"] = pd.to_numeric(movies["year"], errors="coerce")
    year_counts = movies["year"].value_counts().sort_index()

    plt.figure(figsize=(12, 5))
    plt.plot(year_counts.index, year_counts.values)
    plt.title("Number of Movies Released Per Year")
    plt.xlabel("Year")
    plt.ylabel("Number of Movies")
    plt.grid(True)
    plt.show()
else:
    print("Year column not found.")

### 5.2 Most Common Genres

In [ ]:
movies["genre"] = movies["genre"].fillna("")

genre_list = []
for g in movies["genre"]:
    for item in str(g).split(","):
        item = item.strip()
        if item:
            genre_list.append(item)

genre_counts = Counter(genre_list).most_common(15)

labels = [g[0] for g in genre_counts]
values = [g[1] for g in genre_counts]

plt.figure(figsize=(10, 6))
plt.barh(labels, values)
plt.title("Top 15 Most Common Genres")
plt.xlabel("Number of Movies")
plt.gca().invert_yaxis()
plt.show()

### 5.3 Top Directors by Number of Movies

In [ ]:
if "director" in movies.columns:
    director_counts = movies["director"].value_counts().head(15)

    plt.figure(figsize=(10, 6))
    plt.barh(director_counts.index, director_counts.values)
    plt.title("Top 15 Directors with Most Bollywood Movies")
    plt.xlabel("Number of Movies")
    plt.gca().invert_yaxis()
    plt.show()
else:
    print("Director column missing.")

### 5.4 Distribution of Movie Overview Length

In [ ]:
movies["overview_len"] = movies["overview"].astype(str).apply(len)

plt.figure(figsize=(10, 5))
plt.hist(movies["overview_len"], bins=50)
plt.title("Distribution of Movie Overview Length (Text Length)")
plt.xlabel("Length (characters)")
plt.ylabel("Frequency")
plt.show()

## 6. Text Feature Extraction using TF-IDF

**Data Science technique:**

We convert the text **soup** into numerical vectors using **TF-IDF (Term Frequency – Inverse Document Frequency)**.  
This is a common **feature engineering** method in Natural Language Processing (NLP).

In [ ]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["soup"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

## 7. Similarity Computation using Cosine Similarity

**Data Science technique:**

We use **Cosine Similarity** to measure how similar two movies are,  
based on their TF-IDF feature vectors.


In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Cosine similarity matrix shape:", cosine_sim.shape)

## 8. Build a Fast Lookup Index (Title → Row Index)

We create a mapping from **movie title → row index**  
to quickly fetch similarity scores for any selected movie.


In [ ]:
movies = movies.reset_index(drop=True)

indices = pd.Series(
    movies.index,
    index=movies["title"].astype(str).str.lower()
).drop_duplicates()

len(indices), indices.head()

## 9. Recommendation Function (Model Inference Step)

**Data Science technique:**

This function acts as the **inference step of our recommendation model**,  
where we take a movie title as input and output **Top N similar movies**.


In [ ]:
def get_recommendations(title, movies, cosine_sim, indices, n=10):
    """
    Given a movie title, return top N similar movies.
    
    Parameters
    ----------
    title : str
        Movie name to search for.
    movies : pd.DataFrame
        DataFrame containing movie metadata.
    cosine_sim : np.ndarray
        Precomputed cosine similarity matrix.
    indices : pd.Series
        Mapping from movie title (lowercase) to DataFrame index.
    n : int, optional
        Number of recommendations to return (default is 10).
    """
    title_key = str(title).lower().strip()

    if title_key not in indices:
        print(f"Movie '{title}' not found in the dataset.")
        return pd.DataFrame(columns=["title", "year", "genre", "director"])

    idx = indices[title_key]

    # Get similarity scores for this movie with all others
    scores = list(enumerate(cosine_sim[idx]))

    # Sort movies based on similarity score (highest first)
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Skip the first one (it will be the same movie), take next N
    scores = scores[1:n + 1]

    # Extract the movie indices
    movie_indices = [i[0] for i in scores]

    # Columns to show in result
    cols = [c for c in ["title", "year", "genre", "director"] if c in movies.columns]
    return movies.loc[movie_indices, cols].reset_index(drop=True)

## 10. Try the Recommender 🔍

Enter a Bollywood movie title and see the recommended similar movies.

This demonstrates the **application of the trained pipeline** (features + similarity)
to generate useful outputs.


In [ ]:
# Example: change this title to any movie present in your dataset
example_title = "Sholay"  # 👈 replace with any movie name you like

recommendations = get_recommendations(example_title, movies, cosine_sim, indices, n=10)
print(f"Movies similar to: {example_title}\n")
recommendations

### 10.1 Interactive Input (Optional)

In [ ]:
# Uncomment and run this cell if you want to enter a movie name manually

# user_title = input("Enter a Bollywood movie name: ")
# recs = get_recommendations(user_title, movies, cosine_sim, indices, n=10)
# print(f"\nMovies similar to: {user_title}\n")
# display(recs)

## 11. Summary of Data Science Techniques Used

In this notebook, we demonstrated a full **Data Science workflow**:

1. **Data Loading**
   - Loaded CSV data using `pandas.read_csv`.

2. **Data Understanding & Quality Check**
   - Used `.shape`, `.head()`, `.info()`, `.isnull().sum()`, and `.describe()`.

3. **Data Cleaning & Preprocessing**
   - Normalized column names.
   - Selected relevant features (`title`, `genre`, `overview`, etc.).
   - Handled missing values in text columns using imputation (empty strings).

4. **Exploratory Data Analysis (EDA)**
   - Visualized movies per year.
   - Analyzed most common genres.
   - Identified top directors by number of movies.
   - Explored distribution of overview text length.

5. **Feature Engineering**
   - Created a combined text **soup** from multiple columns.
   - Converted text into numerical vectors using **TF-IDF**.

6. **Modeling (Unsupervised / Similarity-based)**
   - Computed **Cosine Similarity** between movie vectors.
   - Implemented a recommendation function to get similar movies.

7. **Model Inference / Application**
   - Took a movie title as input and returned **Top N recommended movies**.

You can now extend this notebook further by:

- Adding rating-based or popularity-based filters.  
- Combining content-based filtering with collaborative filtering.  
- Deploying the recommender as a web app using **Streamlit** or **Gradio**.
